# Phase 2: Destination Research Agent with Real Tools

Replace the stub destination agent with a real LLM-powered agent using:
- **Claude** (ChatAnthropic) for reasoning
- **DuckDuckGo** for web search
- **ReAct loop** (reason -> tool call -> observe -> reason)
- **LangGraph ToolNode** for tool execution

## 1. Setup

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"LLM Provider: {os.getenv('LLM_PROVIDER', 'anthropic')}")

## 2. Test the Tools Individually

In [ ]:
from src.tools.search_tools import web_search, get_destination_info

# Test get_destination_info (local lookup)
print("=== get_destination_info('Bali') ===")
print(get_destination_info.invoke("Bali"))
print()

print("=== get_destination_info('Tokyo') ===")
print(get_destination_info.invoke("Tokyo"))
print()

# Test web_search (live DuckDuckGo)
print("=== web_search('Bali travel guide 2025') ===")
print(web_search.invoke("Bali travel guide 2025"))

## 3. Test the Destination Agent (ReAct with Tools)

The agent uses Claude to reason about the destination, calls tools for data, then synthesizes a research report.

In [ ]:
from src.agents.destination_agent import _run_react_agent

print("=== Researching Bali ===")
print()
result = _run_react_agent("Bali", ["beach", "culture", "adventure"])
print(result)

In [ ]:
print("=== Researching Maldives ===")
print()
result = _run_react_agent("Maldives", ["snorkeling", "spa", "romantic"])
print(result)

In [ ]:
print("=== Researching a lesser-known destination: Luang Prabang, Laos ===")
print()
result = _run_react_agent("Luang Prabang, Laos", ["temples", "nature", "food"])
print(result)

## 4. Run the Full Graph with Real Destination Agent

The destination agent now uses Claude + tools. Other agents remain as stubs (Phase 3-4).

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from src.state import TravelPlanState
from src.supervisor import parse_intent, supervisor_router, aggregate_results
from src.agents.destination_agent import destination_research_agent
from src.agents.flight_agent import flight_search_agent
from src.agents.hotel_agent import hotel_search_agent
from src.agents.activity_agent import activity_itinerary_agent
from src.agents.budget_agent import budget_currency_agent

builder = StateGraph(TravelPlanState)
builder.add_node("intent_parser", parse_intent)
builder.add_node("destination_research", destination_research_agent)
builder.add_node("flight_search", flight_search_agent)
builder.add_node("hotel_search", hotel_search_agent)
builder.add_node("activity_itinerary", activity_itinerary_agent)
builder.add_node("budget_currency", budget_currency_agent)
builder.add_node("aggregator", aggregate_results)

builder.add_edge(START, "intent_parser")
builder.add_conditional_edges("intent_parser", supervisor_router)
builder.add_conditional_edges("destination_research", supervisor_router)
builder.add_conditional_edges("flight_search", supervisor_router)
builder.add_conditional_edges("hotel_search", supervisor_router)
builder.add_conditional_edges("activity_itinerary", supervisor_router)
builder.add_conditional_edges("budget_currency", supervisor_router)
builder.add_edge("aggregator", END)

graph = builder.compile()
print("Graph compiled with REAL destination agent!")
print()

# Run with Bali request
print("Running full graph...")
print()
for step in graph.stream({
    "messages": [
        HumanMessage(
            content="Plan a 5-day beach vacation in Bali for 2 people, "
            "budget $3000, departing from New York. Dates: March 15-20, 2025."
        )
    ],
}):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    agent = node_output.get('current_agent', '')
    is_real = '(REAL - LLM + Tools)' if node_name == 'destination_research' else '(stub)'
    if node_name in ('intent_parser', 'aggregator'):
        is_real = ''
    print(f"--- {node_name} {is_real} ---")
    if 'messages' in node_output and node_output['messages']:
        content = node_output['messages'][-1].content
        if len(content) > 300:
            print(f"  {content[:300]}...")
        else:
            print(f"  {content}")
    print()

## Summary

Phase 2 complete:
- **DuckDuckGo web_search** tool for live destination research
- **get_destination_info** tool for structured travel data
- **ReAct agent** using Claude + LangGraph ToolNode
- **Fallback** to mock data if LLM API is unavailable
- Tested with Bali, Maldives, and a lesser-known destination

Next: Phase 3 — Flight & Hotel agents with mock APIs.